# Exploratory Data Analysis (EDA) - Candidate Discovery

This notebook provides a thorough exploration of the **Redrob Intelligent Candidate Discovery & Ranking Challenge** dataset, focusing on:
1. Schema structural checks
2. Candidate profile distributions (Years of experience, Locations, and Top skills)
3. Platform behavioral/activity signals analysis
4. Detection of temporal anomalies and honeypot traps

In [ ]:
import sys
import os

# Configure backend import paths
sys.path.insert(0, os.path.abspath('../backend'))

# Auto-install any missing dependencies inside the running Jupyter kernel
required_libs = ["numpy", "pandas", "matplotlib", "seaborn", "scikit-learn", "sentence-transformers", "faiss-cpu"]
missing_libs = []
for lib in required_libs:
    try:
        if lib == "faiss-cpu":
            import faiss
        elif lib == "scikit-learn":
            import sklearn
        elif lib == "sentence-transformers":
            import sentence_transformers
        else:
            __import__(lib)
    except ImportError:
        missing_libs.append(lib)

if missing_libs:
    print(f"Installing missing libraries in current kernel: {missing_libs}")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install"] + missing_libs)
    print("Installation complete!")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from app.services.ingestion.ingestion_service import IngestionService
from app.services.ingestion.anomaly_detection import generate_anomaly_report

sns.set_theme(style="darkgrid")
%matplotlib inline

## 1. Load Candidate Profiles

We initialize our ingestion service and load a subset of 10,000 candidate profiles to keep visualization generation fast and responsive.

In [ ]:
service = IngestionService()
status = service.get_candidate_files_status()
print(f"Ingesting from active source: {status.get('active_source')} at {status[status['active_source']]['path']}")

candidates = service.load_all_candidates(limit=10000, validate=True, preprocess=True, as_dict=False)
print(f"Loaded and validated {len(candidates)} candidate profiles.")

## 2. Professional Experience Distribution

In [ ]:
exp_years = [c.profile.years_of_experience for c in candidates]

plt.figure(figsize=(10, 5))
sns.histplot(exp_years, bins=30, kde=True, color="skyblue")
plt.title("Candidate Distribution by Years of Experience")
plt.xlabel("Years of Experience")
plt.ylabel("Candidate Count")
plt.axvline(np.mean(exp_years), color='red', linestyle='--', label=f'Mean: {np.mean(exp_years):.1f} yrs')
plt.axvline(np.median(exp_years), color='green', linestyle='-', label=f'Median: {np.median(exp_years):.1f} yrs')
plt.legend()
plt.show()

## 3. Skills Prevalence Analysis

In [ ]:
skills_counter = Counter()
for c in candidates:
    for s in c.skills:
        skills_counter[s.name] += 1

top_skills_df = pd.DataFrame(skills_counter.most_common(20), columns=["Skill", "Count"])

plt.figure(figsize=(12, 6))
sns.barplot(data=top_skills_df, x="Count", y="Skill", hue="Skill", palette="viridis", legend=False)
plt.title("Top 20 Most Frequent Candidate Skills")
plt.xlabel("Occurrences Count")
plt.ylabel("Skill Standard Name")
plt.show()

## 4. Platform Activity & Behavioral Signals

In [ ]:
completeness = [c.redrob_signals.profile_completeness_score for c in candidates]
response_rates = [c.redrob_signals.recruiter_response_rate for c in candidates]
connections = [c.redrob_signals.connection_count for c in candidates]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(completeness, bins=20, kde=True, ax=axes[0], color="salmon")
axes[0].set_title("Profile Completeness Score")
axes[0].set_xlabel("Completeness %")

sns.histplot(response_rates, bins=20, kde=True, ax=axes[1], color="lightgreen")
axes[1].set_title("Recruiter Response Rate")
axes[1].set_xlabel("Response Rate (0 to 1)")

sns.histplot(connections, bins=20, kde=True, ax=axes[2], color="plum")
axes[2].set_title("Candidate Connection Count")
axes[2].set_xlabel("Connections count")

plt.tight_layout()
plt.show()

## 5. Anomaly Detection Report

We run the dynamic anomaly detection framework to scan the sample candidate records for temporal and logical contradictions.

In [ ]:
report = generate_anomaly_report(candidates)

print("=== Anomaly Report Summary ===")
print(f"Total Profiles Flagged: {report['summary']['flagged_profiles']} out of {report['summary']['total_analyzed']}")
for category, count in report["categories"].items():
    print(f"  {category}: {count} occurrences")

print("\n=== Sample Anomalous Profiles ===")
for hc in report["sample_flagged"][:5]:
    print(f"Candidate ID: {hc['candidate_id']}")
    if hc["temporal_anomalies"]:
        print(f"  Temporal Anomalies: {hc['temporal_anomalies']}")
    if hc["profile_inconsistencies"]:
        print(f"  Profile Inconsistencies: {hc['profile_inconsistencies']}")